# PAV_Publication_Figure

Builds the Pavlovian figure from the analysis pickle and the training-behaviour table.

**Pipeline:** 1. PAV_Preprocessing → 2. PAV_Analysis → **3. PAV_Publication_Figure**

### Inputs (read from this repo)
| Path | Contents |
|---|---|
| `Photometry/Pavlovian/pav_pub_data.pkl` | Output of PAV_Analysis |
| `Photometry/Pavlovian/Behavior/pav_behavior_per_trial.csv` | One row per tone across the three training days |

### Outputs (downloaded at the end)
`PAV_Publication_Figure.svg` and `.png`, and `PAV_Publication_Figure_stats.csv` (every test in the figure, plus the early-vs-late learning tests).

### Statistics
Every test uses one value per mouse (n = mice). Two groups: Welch's t-test. Group x tone duration (or day): mixed ANOVA, then Welch's t per level, Holm-adjusted. Each panel shows the Holm-adjusted p and Hedges' g. The CSV also lists the 95% CI of g, the power to detect the observed effect, the smallest effect detectable with 80% power, and the mice per group needed for 80% power.

### Panels
| Panel | Content |
|---|---|
| **A** | Von Frey data (placeholder) |
| **B** | Task schematic (blank) |
| **C / D** | Success rate / retrieval latency across the 3 training days, binned by pellets earned |
| **E / F** | Timeline and injection-site schematics (blank) |
| **G** | Early recording: tone-onset and pellet-grab heatmaps with PSTHs |
| **H** | Late recording: same |

Statistics on G and H compare SHAM vs SNI on per-mouse peak z and AUC, computed from the plotted traces (tone window 0 to 5 s, grab window −2 to +1 s).

In [ ]:
# @title Setup — imports
import io
import os
import pickle
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import ttest_ind

print("Imports OK.")


Imports OK.


In [ ]:
# @title Source URLs & output path
# Inputs are read from the GitHub repo; outputs are written to the Colab runtime
# and downloaded at the end. Commit the outputs to the repo yourself.

GITHUB_OWNER = "CreedLab-Neuro"   # or "MasBarr" if you point at a personal fork
GITHUB_REPO  = "Manuscript_PMID41279070"
GH_API = f"https://api.github.com/repos/{GITHUB_OWNER}/{GITHUB_REPO}"
PAV    = "Photometry/Pavlovian"

# ── Pin this run to one commit ────────────────────────────────────────────────
# raw.githubusercontent caches BRANCH urls (~5 min), so fetching ".../main/..."
# shortly after a push can serve the PREVIOUS pickle -- and an old pickle plots
# perfectly happily. Resolve main to its commit SHA and fetch by SHA instead.
DATA_SHA = requests.get(f"{GH_API}/commits/main").json()['sha']
GH_RAW   = f"https://raw.githubusercontent.com/{GITHUB_OWNER}/{GITHUB_REPO}/{DATA_SHA}"
PKL_URL  = f"{GH_RAW}/{PAV}/pav_pub_data.pkl"
BEH_URL  = f"{GH_RAW}/{PAV}/Behavior/pav_behavior_per_trial.csv"   # panels C and D
print(f"Data version (main @ commit): {DATA_SHA[:10]}")

OUTPUT_DIR = '/content/PAV_Figures'                 # Colab-local; downloaded at the end
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Pickle   : {PKL_URL}\nBehavior : {BEH_URL}\nOutput   : {OUTPUT_DIR}")


In [ ]:
# @title Load the pickle from GitHub — the only input this notebook has
_pr = requests.get(PKL_URL)
if _pr.status_code != 200:
    raise SystemExit(
        f"[STOP] could not fetch {PKL_URL} (HTTP {_pr.status_code}).\n"
        f"       Run PAV_Analysis and commit pav_pub_data.pkl to {PAV}/ first.")
D = pickle.loads(_pr.content)

# trial_evt holds the per-trial traces behind the G/H heatmaps. Pickles made
# before it was added cannot draw the retrieval heatmaps -- stop rather than
# quietly draw half a figure.
if 'trial_evt' not in D:
    raise SystemExit(
        "[STOP] pickle has no 'trial_evt' -- it predates the per-trial retrieval traces.\n"
        "       Re-run PAV_Analysis and commit the new pav_pub_data.pkl.")

event_time   = D['event_time']      # x-axis shared by psth_animals and trial_evt
psth_animals = D['psth_animals']    # {event: {group: {session: [per-animal traces]}}}
trial_evt    = D['trial_evt']       # {event: {session: {group: [(animal, trace)]}}}
DA_WINDOWS   = D['DA_WINDOWS']      # {'cue': (a, b), 'grab': (a, b)} in s from the event
GROUP_COLORS = dict(D['GROUP_COLORS'])

print("Loaded pickle. Keys:", list(D.keys()))

# ── Training behaviour for panels C and D ────────────────────────────────────
# One row per tone across the 3 training days, both cohorts (OFP + OFP2). Produced
# by pav_behavior_summary.py; animal_id carries the cohort prefix because OFP2's
# zero-padded IDs (M01) collide with OFP's (M1).
_br = requests.get(BEH_URL)
if _br.status_code != 200:
    raise SystemExit(f"[STOP] could not fetch {BEH_URL} (HTTP {_br.status_code}).")
beh = pd.read_csv(io.StringIO(_br.text))
beh['pellets_earned'] = (beh.groupby(['group', 'animal_id', 'day'])
                            .outcome.transform(lambda s: (s == 'success').cumsum()))
print(f"Behaviour: {len(beh):,} trials, "
      f"{beh.groupby('group').animal_id.nunique().to_dict()} animals, days {sorted(beh.day.unique())}")


Loaded pickle. Keys: ['pav_df', 'event_time', 'psth_animals', 'trial_psths', 'trial_evt', 'trial_dist_tone', 'animal_dist_tone', 'hm_time', 'delta_df', 'DA_WINDOWS', 'GROUPS', 'SESSIONS', 'GROUP_COLORS', 'PRES_FS']
Behaviour: 15,393 trials, {'SHAM': 14, 'SK3': 8, 'SNI': 13} animals, days [np.int64(1), np.int64(2), np.int64(3)]


In [ ]:
# @title Constants, publication style & helpers

# Publication colors -- match the VT figure, override what the pickle carries
GROUP_COLORS['SHAM'] = '#2166AC'   # blue
GROUP_COLORS['SNI']  = '#C0392B'   # red

PUB_GROUPS = ['SHAM', 'SNI']       # add 'SK3' here to overlay the rescue group
EVENTS     = [('Cue', 'cue', 'Time from\nTone Onset (s)'),
              ('PelletGrab', 'grab', 'Time from Pellet\nRetrieval (s)')]
XLIM       = (-2, 2)               # plotted window, s from the event

# Per-trace baseline subtraction. None = plot the 25th-percentile z-score as-is,
# which is what the VT figure does (traces then sit a few z above 0 at rest).
# Set e.g. (-2.0, -0.5) to subtract each trace's mean over that window instead.
BASELINE_WIN = None

HM_VMIN, HM_VMAX = 0, 20           # heatmap colour limits (z), as in the VT figure
LW_MEAN = 3.0

PRES_FS = {'title': 21, 'label': 18, 'tick': 16, 'legend': 16, 'annot': 13}
plt.rcParams.update({
    'axes.linewidth': 2.5,
    'xtick.major.width': 2.5, 'ytick.major.width': 2.5,
    'xtick.major.size': 6,    'ytick.major.size': 6,
    'font.size': 17,
    'svg.fonttype': 'none',   # keep text as text in the SVG, so it stays editable
})

def baseline(M):
    """Apply BASELINE_WIN to every row of a (traces x time) array."""
    M = np.asarray(M, float)
    if BASELINE_WIN is None: return M
    w = (event_time >= BASELINE_WIN[0]) & (event_time <= BASELINE_WIN[1])
    return M - np.nanmean(M[:, w], axis=1, keepdims=True)

def mean_sem(M):
    gm = np.nanmean(M, axis=0)
    gs = np.nanstd(M, axis=0, ddof=1) / np.sqrt(M.shape[0])
    return gm, gs

def despine(ax):
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

def blank(ax):
    for sp in ax.spines.values(): sp.set_visible(False)
    ax.set_xticks([]); ax.set_yticks([])

def window_metrics(M, win):
    """Per-animal (peak z, AUC in z*s) over `win`, from the plotted traces."""
    w = (event_time >= win[0]) & (event_time <= win[1])
    return np.nanmax(M[:, w], axis=1), np.trapezoid(M[:, w], event_time[w], axis=1)

def p_text(p):
    if not np.isfinite(p): return 'n/a'
    star = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
    return f'p = {p:.3f} {star}' if p >= 0.001 else f'p < 0.001 {star}'

print("Style + helpers ready.")


Style + helpers ready.


In [ ]:
# @title Statistics helpers — animal-level tests, effect size, power
# Every test in this figure is run on ONE value per mouse (n = mice, not trials).
#   Two groups, one value per mouse  -> Welch's t-test
#   Two groups x tone duration       -> mixed ANOVA (group between, duration within),
#                                       then Welch's t per duration, Holm-adjusted
#   Effect size  -> Hedges' g on mouse means (positive = second group higher), 95% CI
#   Power        -> power to detect the observed g with these n; the smallest g these n
#                   detect with 80% power; mice per group needed for 80% power at the
#                   observed g (two-sided alpha = 0.05)
# The primary test of each outcome is Holm-adjusted across all outcomes in the figure.
import sys, subprocess
try:
    import pingouin as pg
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pingouin'], check=True)
    import pingouin as pg
from scipy.stats import ttest_ind
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.power import TTestIndPower

ALPHA, TARGET_POWER = 0.05, 0.80
_POWER = TTestIndPower()

def hedges_g(a, b):
    """Hedges' g = (mean b - mean a) / pooled SD, small-sample corrected, with 95% CI."""
    a, b = np.asarray(a, float), np.asarray(b, float)
    a, b = a[np.isfinite(a)], b[np.isfinite(b)]
    n1, n2 = len(a), len(b)
    sp = np.sqrt(((n1 - 1) * a.var(ddof=1) + (n2 - 1) * b.var(ddof=1)) / (n1 + n2 - 2))
    g  = (b.mean() - a.mean()) / sp * (1 - 3 / (4 * (n1 + n2) - 9))
    se = np.sqrt((n1 + n2) / (n1 * n2) + g ** 2 / (2 * (n1 + n2)))
    return g, g - 1.96 * se, g + 1.96 * se

def power_row(g, n1, n2):
    """Observed power, minimum detectable g (80%), and mice/group needed for 80%."""
    r = n2 / n1
    mde = _POWER.solve_power(nobs1=n1, ratio=r, alpha=ALPHA, power=TARGET_POWER)
    if abs(g) < 0.05:
        return dict(power_obs=ALPHA, mde_g=mde, n_per_group_80=np.inf)
    pw  = _POWER.power(effect_size=abs(g), nobs1=n1, ratio=r, alpha=ALPHA)
    need = _POWER.solve_power(effect_size=abs(g), ratio=1.0, alpha=ALPHA, power=TARGET_POWER)
    return dict(power_obs=pw, mde_g=mde, n_per_group_80=int(np.ceil(need)))

def welch_row(a, b):
    """Welch's t on mouse values + effect size + power."""
    a, b = np.asarray(a, float), np.asarray(b, float)
    a, b = a[np.isfinite(a)], b[np.isfinite(b)]
    t = ttest_ind(a, b, equal_var=False)
    g, lo, hi = hedges_g(a, b)
    return dict(n1=len(a), n2=len(b), mean1=a.mean(), mean2=b.mean(), sd1=a.std(ddof=1),
                sd2=b.std(ddof=1), t=t.statistic, df=getattr(t, 'df', np.nan), p=t.pvalue,
                g=g, g_lo=lo, g_hi=hi, **power_row(g, len(a), len(b)))

def mixed_anova_row(long, dv, g1, g2):
    """Mixed ANOVA (group x duration) on a long table with columns animal, group, dur, dv.
    Only mice with all durations enter (a requirement of the ANOVA). Effect size and power
    use each mouse's mean over durations, which is what the group main effect tests."""
    d = long[long['group'].isin([g1, g2])].dropna(subset=[dv])
    d = d.groupby('animal').filter(lambda x: x['dur'].nunique() == d['dur'].nunique())
    aov = pg.mixed_anova(data=d, dv=dv, within='dur', between='group', subject='animal')
    pcol = 'p-unc' if 'p-unc' in aov.columns else 'p_unc'
    G, I = aov[aov.Source == 'group'].iloc[0], aov[aov.Source == 'Interaction'].iloc[0]
    m = d.groupby(['animal', 'group'])[dv].mean().reset_index()
    a, b = m[m.group == g1][dv].values, m[m.group == g2][dv].values
    g, lo, hi = hedges_g(a, b)
    return dict(n1=len(a), n2=len(b), mean1=a.mean(), mean2=b.mean(), sd1=a.std(ddof=1),
                sd2=b.std(ddof=1), F=G['F'], df=f"{int(G['DF1'])},{int(G['DF2'])}", p=G[pcol],
                np2=G['np2'], p_interaction=I[pcol], g=g, g_lo=lo, g_hi=hi,
                **power_row(g, len(a), len(b)))

def holm(pvals):
    p = np.asarray(pvals, float); out = np.full_like(p, np.nan); ok = np.isfinite(p)
    if ok.any(): out[ok] = multipletests(p[ok], method='holm')[1]
    return out

def fmt_p(p):
    if not np.isfinite(p): return 'p = n/a'
    return 'p < 0.001' if p < 0.001 else f'p = {p:.3f}' if p < 0.01 else f'p = {p:.2f}'

def p_g(row, adj=True):
    """'p = 0.21, g = 0.60' for a panel annotation (Holm-adjusted p when available)."""
    p = row.get('p_holm', row['p']) if adj else row['p']
    return f"{fmt_p(p)}, g = {row['g']:.2f}"

print("Stats helpers ready.")


In [ ]:
# @title Statistics — every test in the figure (printed and saved as a CSV)
# One value per mouse enters every test.
#   G / H  : SHAM vs SNI peak z and AUC per event and recording (Welch's t on mouse values,
#            from exactly the traces that are plotted; windows from DA_WINDOWS)
#   C / D  : SHAM vs SNI training behaviour, group x day mixed ANOVA on each mouse's daily
#            success rate and mean latency of successful trials
#   Learning (early -> late recording): session x group mixed ANOVA on the same DA values;
#            the session effect asks whether responses change with training, the interaction
#            whether that change differs between groups
# Group tests are Holm-adjusted together (G/H and C/D); the learning tests separately.
FIG_NAME  = 'PAV_Publication_Figure'
G1, G2    = PUB_GROUPS[0], PUB_GROUPS[1]
_EV_KEY   = {ev: k for ev, k, _ in EVENTS}

def animal_means(rows):
    """[(animal, trace), ...] -> {animal: mean trace}."""
    by = {}
    for a, tr in rows:
        by.setdefault(a, []).append(np.asarray(tr, float))
    return {a: np.nanmean(np.vstack(v), axis=0) for a, v in by.items()}

# Per-mouse peak / AUC, keyed by animal so the two recordings can be paired
_da = []
for ev, k, _ in EVENTS:
    w = (event_time >= DA_WINDOWS[k][0]) & (event_time <= DA_WINDOWS[k][1])
    for ses in ('01', '02'):
        for grp in PUB_GROUPS:
            for a, tr in animal_means(trial_evt[ev][ses][grp]).items():
                tr = baseline(tr[None, :])[0]
                _da.append(dict(animal=a, group=grp, session=ses, event=ev,
                                peak=np.nanmax(tr[w]), auc=np.trapezoid(tr[w], event_time[w])))
DA = pd.DataFrame(_da)

# Check: these must be the traces the PSTH panels plot (psth_animals)
for ev, k, _ in EVENTS:
    for ses in ('01', '02'):
        for grp in PUB_GROUPS:
            n_plot = len(psth_animals[ev][grp][ses])
            n_here = ((DA.event == ev) & (DA.session == ses) & (DA.group == grp)).sum()
            assert n_plot == n_here, f"{ev} {ses} {grp}: {n_plot} plotted vs {n_here} mice"

_rows = []
for ses, rec in (('01', 'Early'), ('02', 'Late')):
    for ev, k, _ in EVENTS:
        for m, lab in (('peak', 'peak z'), ('auc', 'AUC (z*s)')):
            d = DA[(DA.session == ses) & (DA.event == ev)]
            _rows.append(dict(panel='G' if ses == '01' else 'H', outcome=f'{rec} {ev} {lab}',
                              event=ev, session=ses, metric=m, contrast=f'{G1} vs {G2}',
                              family='group', test='Welch t',
                              **welch_row(d[d.group == G1][m], d[d.group == G2][m])))

# Training behaviour: one value per mouse per day
_b = beh[beh.group.isin(PUB_GROUPS)].copy()
_b['success'] = 100.0 * (_b.outcome == 'success')
BEH_DAY = (_b.groupby(['group', 'animal_id', 'day']).success.mean().rename('success').to_frame()
             .join(_b[_b.outcome == 'success'].groupby(['group', 'animal_id', 'day'])
                     .latency_from_tone_onset_s.mean().rename('latency')).reset_index()
             .rename(columns={'animal_id': 'animal', 'day': 'dur'}))
for panel, dv, lab in (('C', 'success', 'Success rate (%)'), ('D', 'latency', 'Retrieval latency (s)')):
    _rows.append(dict(panel=panel, outcome=lab, contrast=f'{G1} vs {G2}', family='group',
                      test='Mixed ANOVA, group effect (group x day)',
                      **mixed_anova_row(BEH_DAY, dv, G1, G2)))

# Learning: early vs late recording, session x group
from statsmodels.stats.power import TTestPower
for ev, k, _ in EVENTS:
    for m, lab in (('peak', 'peak z'), ('auc', 'AUC (z*s)')):
        d = DA[DA.event == ev].pivot_table(index=['animal', 'group'], columns='session', values=m).dropna()
        long = d.stack().rename(m).reset_index()
        aov = pg.mixed_anova(data=long, dv=m, within='session', between='group', subject='animal')
        pcol = 'p-unc' if 'p-unc' in aov.columns else 'p_unc'
        S_, I_ = aov[aov.Source == 'session'].iloc[0], aov[aov.Source == 'Interaction'].iloc[0]
        chg = d['02'] - d['01']
        dz = chg.mean() / chg.std(ddof=1)            # within-mouse effect size (Cohen's dz)
        tp = TTestPower()
        _rows.append(dict(panel='G vs H', outcome=f'Learning: {ev} {lab}', event=ev, metric=m,
                          contrast='early vs late (all mice)', family='learning',
                          test='Mixed ANOVA, session effect (session x group)',
                          n1=len(d), mean1=d['01'].mean(), mean2=d['02'].mean(),
                          F=S_['F'], df=f"{int(S_['DF1'])},{int(S_['DF2'])}", p=S_[pcol], np2=S_['np2'],
                          p_interaction=I_[pcol], g=dz,
                          power_obs=tp.power(effect_size=abs(dz), nobs=len(d), alpha=ALPHA),
                          mde_g=tp.solve_power(nobs=len(d), alpha=ALPHA, power=TARGET_POWER),
                          n_per_group_80=int(np.ceil(tp.solve_power(effect_size=max(abs(dz), 0.05),
                                                                     alpha=ALPHA, power=TARGET_POWER)))))

STATS = pd.DataFrame(_rows)
for fam in ('group', 'learning'):
    m = STATS.family == fam
    STATS.loc[m, 'p_holm'] = holm(STATS.loc[m, 'p'])

def stat_row(outcome):
    return STATS[STATS.outcome == outcome].iloc[0].to_dict()

pd.set_option('display.width', 220); pd.set_option('display.max_columns', 30)
_show = ['panel', 'outcome', 'contrast', 'test', 'n1', 'n2', 'mean1', 'mean2', 'p', 'p_holm',
         'p_interaction', 'np2', 'g', 'g_lo', 'g_hi', 'power_obs', 'mde_g', 'n_per_group_80']
print(STATS[[c for c in _show if c in STATS]].round(3).to_string(index=False))
print("\ng: Hedges' g (SNI minus SHAM) for group tests; Cohen's dz (late minus early) for learning.")
out_csv = f"{OUTPUT_DIR}/{FIG_NAME}_stats.csv"
STATS.to_csv(out_csv, index=False)
print(f"Saved: {out_csv}")


In [ ]:
# @title Build the figure & save (SVG + PNG)

fig = plt.figure(figsize=(26, 13))

# Two independent rows (different column counts), like the VT figure
gs_r1 = gridspec.GridSpec(1, 4, figure=fig, left=0.04, right=0.98, top=0.96, bottom=0.62,
                          width_ratios=[0.9, 1.2, 1.6, 1.6], wspace=0.35)
gs_r2 = gridspec.GridSpec(1, 3, figure=fig, left=0.04, right=0.95, top=0.50, bottom=0.07,
                          width_ratios=[0.8, 2.0, 2.0], wspace=0.22)

axd = {}
axd['A'] = fig.add_subplot(gs_r1[0, 0])
axd['B'] = fig.add_subplot(gs_r1[0, 1])
for key, col in (('C', 2), ('D', 3)):             # three training days each
    sub = gridspec.GridSpecFromSubplotSpec(1, 3, subplot_spec=gs_r1[0, col], wspace=0.12)
    for d in range(3):
        axd[f'{key}{d + 1}'] = fig.add_subplot(sub[d])

sub_ef = gridspec.GridSpecFromSubplotSpec(2, 1, subplot_spec=gs_r2[0, 0], hspace=0.25)
axd['E'] = fig.add_subplot(sub_ef[0])
axd['F'] = fig.add_subplot(sub_ef[1])

# G and H: two event columns, heatmap above PSTH
for key, col in (('G', 1), ('H', 2)):
    sub = gridspec.GridSpecFromSubplotSpec(2, 2, subplot_spec=gs_r2[0, col],
                                           height_ratios=[1, 1.35], hspace=0.08, wspace=0.28)
    for j, (ev, _, _) in enumerate(EVENTS):
        axd[f'{key}_hm_{ev}'] = fig.add_subplot(sub[0, j])
        axd[f'{key}_ps_{ev}'] = fig.add_subplot(sub[1, j])

_PANEL_LABELS = {'A': 'A', 'B': 'B', 'C1': 'C', 'D1': 'D', 'E': 'E', 'F': 'F',
                 'G_hm_Cue': 'G', 'H_hm_Cue': 'H'}
for key, lbl in _PANEL_LABELS.items():
    axd[key].text(-0.22 if key[0] in 'CDGH' else -0.08, 1.10, lbl,
                  transform=axd[key].transAxes, fontsize=24, fontweight='bold', va='top')

# ── A: Von Frey placeholder ───────────────────────────────────────────────────
blank(axd['A'])
axd['A'].text(0.5, 0.5, 'Von Frey\nData', transform=axd['A'].transAxes,
              ha='center', va='center', fontsize=PRES_FS['title'], color='#888888')

# ── B, E, F: left empty for the schematics ───────────────────────────────────
for k in ('B', 'E', 'F'):
    blank(axd[k])

# ── C / D: training behaviour across the 3 days, binned by pellets earned ────
# One value per animal per bin, then mean +/- SEM ACROSS ANIMALS, so n is animals.
# Panel D reproduces the published "meanLatency": the mean latency of SUCCESSFUL
# trials, measured from tone ONSET. Including the misses turns the first bin into
# a several-minute median and blows the axis out.
PELLET_BINS = [(1, 10), (11, 20), (21, 30), (31, 40), (41, 50)]
BIN_X       = [np.mean(b) for b in PELLET_BINS]
MIN_TRIALS  = 3                       # an animal needs this many trials in a bin

def behaviour_bin(day, grp, lo, hi, what):
    x = beh[(beh.day == day) & (beh.group == grp) &
            (beh.pellets_earned >= lo) & (beh.pellets_earned <= hi)]
    if what == 'latency':
        x = x[x.outcome == 'success']
    out = []
    for _, g in x.groupby('animal_id'):
        if len(g) < MIN_TRIALS: continue
        out.append(100.0 * (g.outcome == 'success').mean() if what == 'success'
                   else float(g.latency_from_tone_onset_s.mean()))
    return np.array(out, dtype=float)

for key, what, ylab in (('C', 'success', 'Success rate (%)'),
                        ('D', 'latency', 'Retrieval latency (s)')):
    for day in range(1, 4):
        ax = axd[f'{key}{day}']
        for grp in PUB_GROUPS:
            m, e = [], []
            for lo, hi in PELLET_BINS:
                v = behaviour_bin(day, grp, lo, hi, what)
                m.append(np.nan if not len(v) else v.mean())
                e.append(np.nan if len(v) < 2 else v.std(ddof=1) / np.sqrt(len(v)))
            m, e = np.array(m), np.array(e)
            ax.plot(BIN_X, m, color=GROUP_COLORS[grp], lw=LW_MEAN, label=grp)
            ax.fill_between(BIN_X, m - e, m + e, color=GROUP_COLORS[grp], alpha=0.18)
        ax.set_title(f'Day {day}', fontsize=PRES_FS['label'])
        ax.set_xticks([10, 30, 50]); ax.set_xlim(2, 53)
        ax.tick_params(labelsize=PRES_FS['tick'])
        if key == 'C': ax.set_ylim(0, 100); ax.set_yticks([0, 50, 100])
        if day == 1: ax.set_ylabel(ylab, fontsize=PRES_FS['label'])
        else:        ax.set_yticklabels([])
        if day == 2: ax.set_xlabel('Pellets Earned', fontsize=PRES_FS['label'])
        despine(ax)
    if key == 'D':                    # one shared y-range across the three days
        lo = min(a.get_ylim()[0] for a in (axd['D1'], axd['D2'], axd['D3']))
        hi = max(a.get_ylim()[1] for a in (axd['D1'], axd['D2'], axd['D3']))
        for a in (axd['D1'], axd['D2'], axd['D3']): a.set_ylim(lo, hi)
axd['C3'].legend(frameon=False, fontsize=PRES_FS['legend'], loc='lower right')

# ── G / H: heatmaps + PSTHs ───────────────────────────────────────────────────
tw = (event_time >= XLIM[0]) & (event_time <= XLIM[1])

def draw_recording(key, session, title):
    for j, (ev, win_key, xlab) in enumerate(EVENTS):
        ax_hm, ax_ps = axd[f'{key}_hm_{ev}'], axd[f'{key}_ps_{ev}']

        # Heatmap: one row per trial, SHAM block over SNI block, each sorted by
        # peak z inside the plotted window (strongest responses on top).
        blocks, sizes = [], []
        for grp in PUB_GROUPS:
            rows = trial_evt[ev][session][grp]
            M = baseline([r[1] for r in rows]) if rows else np.empty((0, len(event_time)))
            if len(M): M = M[np.argsort(-np.nanmax(M[:, tw], axis=1))]
            blocks.append(M[:, tw]); sizes.append(len(M))
        sep = np.full((max(1, sum(sizes) // 60), tw.sum()), np.nan)
        mat = np.vstack([b for pair in zip(blocks, [sep] * len(blocks)) for b in pair][:-1])
        ax_hm.imshow(mat, aspect='auto', interpolation='nearest', cmap='magma',
                     vmin=HM_VMIN, vmax=HM_VMAX, origin='upper',
                     extent=[event_time[tw][0], event_time[tw][-1], mat.shape[0] - 0.5, -0.5])
        ax_hm.axvline(0, color='white', lw=2.5, ls='--', alpha=0.95)
        for sp in ax_hm.spines.values(): sp.set_visible(False)
        ax_hm.set_xticks([])
        centres, pos = [], 0
        for n in sizes:
            centres.append(pos + n / 2); pos += n + sep.shape[0]
        ax_hm.set_yticks(centres)
        ax_hm.set_yticklabels(PUB_GROUPS, rotation=90, va='center', fontsize=PRES_FS['tick'] - 2)
        ax_hm.tick_params(axis='y', length=0)
        for lbl, grp in zip(ax_hm.get_yticklabels(), PUB_GROUPS):
            lbl.set_color(GROUP_COLORS[grp]); lbl.set_fontweight('bold')

        # PSTH: per-animal means -> group mean +/- SEM (n = animals)
        per_grp = {}
        for grp in PUB_GROUPS:
            traces = psth_animals[ev][grp][session]
            if not traces: continue
            M = baseline(np.vstack(traces)); per_grp[grp] = M
            gm, gs = mean_sem(M)
            ax_ps.plot(event_time, gm, color=GROUP_COLORS[grp], lw=LW_MEAN, label=grp)
            ax_ps.fill_between(event_time, gm - gs, gm + gs, color=GROUP_COLORS[grp], alpha=0.18)
        ax_ps.axvline(0, color='k', lw=2.0, ls='--', ymax=0.72)   # stops below the stats text
        ax_ps.axhline(0, color='gray', lw=0.6, ls=':', alpha=0.5)
        ax_ps.set_xlim(*XLIM)
        ax_ps.set_xticks([XLIM[0], 0, XLIM[1]])
        ax_ps.set_xlabel(xlab, fontsize=PRES_FS['label'])
        ax_ps.tick_params(labelsize=PRES_FS['tick'])
        if j == 0:
            ax_ps.set_ylabel('GRAB$^{DA}$ (z-score)', fontsize=PRES_FS['label'])
        if key == 'G' and j == 0:          # one legend for the whole figure
            ax_ps.legend(frameon=False, fontsize=PRES_FS['legend'], loc='upper left',
                         handlelength=1.2, borderaxespad=0.2)
        despine(ax_ps)

        # Stats from the table computed above: Holm-adjusted p and Hedges' g (n = mice)
        rec = 'Early' if session == '01' else 'Late'
        if len(per_grp) == 2:
            rp, ra = stat_row(f'{rec} {ev} peak z'), stat_row(f'{rec} {ev} AUC (z*s)')
            n_txt = ' / '.join(f'{g} {len(per_grp[g])}' for g in PUB_GROUPS)
            ax_ps.text(0.98, 0.98, f'n = {n_txt}\npeak: {p_g(rp)}\nAUC: {p_g(ra)}',
                       transform=ax_ps.transAxes, ha='right', va='top',
                       fontsize=PRES_FS['annot'], color='#333333', linespacing=1.3)

    # Recording title centred over both event columns
    l = axd[f'{key}_hm_{EVENTS[0][0]}'].get_position()
    r = axd[f'{key}_hm_{EVENTS[-1][0]}'].get_position()
    fig.text((l.x0 + r.x1) / 2, l.y1 + 0.012, title, ha='center', va='bottom',
             fontsize=PRES_FS['title'])

draw_recording('G', '01', 'Early Recording')
draw_recording('H', '02', 'Late Recording')

# One y-range for all four PSTHs so early and late are directly comparable
_ps = [axd[k] for k in axd if '_ps_' in k]
_lo = min(a.get_ylim()[0] for a in _ps); _hi = max(a.get_ylim()[1] for a in _ps)
_hi = _lo + (_hi - _lo) * 1.35          # keep the top quarter clear for the stats text
for a in _ps:
    a.set_ylim(_lo, _hi)
    a.yaxis.set_major_locator(plt.MaxNLocator(integer=True, nbins=4))
    if a not in (axd['G_ps_Cue'], axd['H_ps_Cue']): a.set_yticklabels([])

# Shared colorbar to the right of H
_im  = axd['H_hm_PelletGrab'].images[0]
_pos = axd['H_hm_PelletGrab'].get_position()
_cax = fig.add_axes([_pos.x1 + 0.008, _pos.y0, 0.008, _pos.height])
_cb  = fig.colorbar(_im, cax=_cax)
_cb.set_label('z-score', fontsize=PRES_FS['label'] - 3)
_cb.ax.tick_params(labelsize=PRES_FS['tick'] - 3)

# ── Statistics footnote ───────────────────────────────────────────────────────
for key, out in (('C2', 'Success rate (%)'), ('D2', 'Retrieval latency (s)')):
    r = stat_row(out)
    axd[key].text(0.5, -0.20, f"{G1} vs {G2}: F({r['df']}) = {r['F']:.2f}, {p_g(r)}",
                  transform=axd[key].transAxes, ha='center', va='top', fontsize=PRES_FS['annot'])
_r0 = STATS[STATS.family == 'group'].iloc[0]
fig.text(0.04, -0.02,
         f"Statistics on mouse values (n = mice). G, H: Welch's t-test on each mouse's peak z and AUC "
         f"(cue 0-5 s, retrieval -2 to 1 s). C, D: group x day mixed ANOVA on daily mouse means. "
         f"p values Holm-adjusted across all 10 group tests. g = Hedges' g ({G2} minus {G1}).\n"
         f"Power: with n = {int(_r0['n1'])} vs {int(_r0['n2'])} mice, a two-sided test at alpha = 0.05 "
         f"has 80% power only for g >= {_r0['mde_g']:.2f}. Learning (early vs late) tests: {FIG_NAME}_stats.csv.",
         ha='left', va='top', fontsize=12, color='#333333')

# ── Save (SVG + PNG, Colab-local) and show ────────────────────────────────────
out_svg = f"{OUTPUT_DIR}/PAV_Publication_Figure.svg"
out_png = f"{OUTPUT_DIR}/PAV_Publication_Figure.png"
plt.savefig(out_svg, format='svg', bbox_inches='tight')
plt.savefig(out_png, dpi=150, bbox_inches='tight')
print(f"\nSaved: {out_svg}\nSaved: {out_png}")
plt.show()


In [ ]:
# @title Download the figure (SVG + PNG)
# Colab-local output does not survive the runtime. Keep the SVG -- it is vector,
# so it stays sharp at any journal's figure width.
from google.colab import files

for _f in (out_svg, out_png, out_csv):
    print(f"{_f}  ({os.path.getsize(_f)/1e6:.2f} MB)")
    files.download(_f)
